# Blocked crossings vs Form 71 inventory

Join blocked crossing events to the Form 71 inventory using `Crossing ID`, then compare blocked vs unblocked crossings and inspect density/sparsity.

In [2]:
import pandas as pd
import plotly.express as px

from IPython.display import display
from analyze_blocked_crossings import (
    DEFAULT_BLOCKED_CSV,
    DEFAULT_BLOCKED_XLSX,
    DEFAULT_INVENTORY,
    DEFAULT_OUTPUT,
    clean_columns,
    clean_ids,
    make_comparison_table,
    preprocess_blocked_crossings,
    read_csv,
    save_bar,
    save_group_boxplot,
    save_histogram,
    summarize_counts,
    summarize_state_density,
    to_numeric,
)

output_dir = DEFAULT_OUTPUT
output_dir.mkdir(exist_ok=True)


In [3]:
blocked = preprocess_blocked_crossings(DEFAULT_BLOCKED_XLSX, DEFAULT_BLOCKED_CSV)
inventory = clean_columns(read_csv(DEFAULT_INVENTORY, low_memory=False))

blocked['Crossing ID'] = clean_ids(blocked['Crossing ID'])
inventory['Crossing ID'] = clean_ids(inventory['Crossing ID'])

inventory['Revision Date'] = pd.to_datetime(inventory.get('Revision Date'), errors='coerce')
inventory = inventory.sort_values(['Crossing ID', 'Revision Date']).drop_duplicates('Crossing ID', keep='last')

blocked = blocked.dropna(subset=['Crossing ID']).copy()

blocked_counts = blocked.groupby('Crossing ID').size().rename('blocked_event_count').reset_index()
inventory_with_counts = inventory.merge(blocked_counts, on='Crossing ID', how='left')
inventory_with_counts['blocked_event_count'] = inventory_with_counts['blocked_event_count'].fillna(0).astype(int)
inventory_with_counts['is_blocked'] = inventory_with_counts['blocked_event_count'] > 0

blocked_joined = blocked.merge(inventory, on='Crossing ID', how='left', suffixes=('_blocked', '_inventory'))
blocked_joined.to_csv(output_dir / 'blocked_events_joined_to_inventory.csv', index=False)

state_density = summarize_state_density(inventory_with_counts)
comparison_features = [
    'Annual Average Daily Traffic Count',
    'Annual Average Daily Traffic Year',
    'Total Daylight Thru Trains',
    'Total Nighttime Thru Trains',
    'Total Switching Trains',
    'Total Transit Trains',
    'Number Of Main Tracks',
    'Number Of Siding Tracks',
    'Number Of Yard Tracks',
    'Maximum Timetable Speed',
    'Highway Speed Limit',
]
comparison_table = make_comparison_table(inventory_with_counts, 'is_blocked', comparison_features)


In [4]:
summary = pd.DataFrame([
    ('blocked_events_total', len(blocked)),
    ('blocked_crossing_ids', blocked['Crossing ID'].nunique()),
    ('inventory_crossings_total', len(inventory)),
    ('inventory_crossings_with_blocked_events', int(inventory_with_counts['is_blocked'].sum())),
    ('inventory_share_with_blocked_events_pct', round(inventory_with_counts['is_blocked'].mean() * 100, 4)),
], columns=['metric', 'value'])

display(summary)
display(summarize_counts(inventory_with_counts.loc[inventory_with_counts['is_blocked'], 'blocked_event_count']))
display(state_density.head(10))
display(comparison_table.head(20))


,metric,value
0,blocked_events_total,26131.0000
1,blocked_crossing_ids,7037.0000
2,inventory_crossings_total,438668.0000
3,inventory_crossings_with_blocked_events,7037.0000
4,inventory_share_with_blocked_events_pct,1.6042


,metric,value
0,count,7037.000000
1,mean,3.713372
2,std,17.795571
3,min,1.000000
4,50%,1.000000
5,75%,2.000000
6,90%,6.000000
7,95%,10.000000
8,99%,37.640000
9,max,937.000000


,State Name,total_crossings,blocked_crossings,blocked_events,blocked_crossing_share_pct,blocked_events_per_blocked_crossing
0,ARIZONA,2396,133,468,5.550918,3.518797
1,TEXAS,27540,1056,7550,3.834423,7.149621
2,UTAH,3080,95,476,3.084416,5.010526
3,FLORIDA,9892,292,817,2.951880,2.797945
4,DISTRICT OF COLUMBIA,200,5,20,2.500000,4.000000
5,OKLAHOMA,9116,223,762,2.446248,3.417040
6,INDIANA,16905,392,957,2.318841,2.441327
7,ARKANSAS,6960,155,309,2.227011,1.993548
8,MICHIGAN,13494,300,753,2.223210,2.510000
9,TENNESSEE,8972,195,939,2.173428,4.815385


,feature,group,n,mean,median,p25,p75
0,Annual Average Daily Traffic Count,blocked,6613,5777.998185,2708.0,594.0,7849.0
1,Annual Average Daily Traffic Count,unblocked,257893,2338.362383,280.0,40.0,1560.0
2,Annual Average Daily Traffic Year,blocked,6781,2010.204837,2016.0,2006.0,2022.0
3,Annual Average Daily Traffic Year,unblocked,337459,1991.605185,1988.0,1970.0,2010.0
4,Highway Speed Limit,blocked,5713,34.577105,35.0,30.0,40.0
5,Highway Speed Limit,unblocked,129400,31.203076,30.0,25.0,40.0
6,Maximum Timetable Speed,blocked,7023,41.751958,40.0,20.0,60.0
7,Maximum Timetable Speed,unblocked,429449,23.625383,20.0,6.0,40.0
8,Number Of Main Tracks,blocked,7023,1.191514,1.0,1.0,2.0
9,Number Of Main Tracks,unblocked,430200,0.714000,1.0,0.0,1.0


In [5]:
reason_counts = blocked['Reason'].value_counts(dropna=False)
state_counts = blocked['State'].value_counts(dropna=False)

save_histogram(inventory_with_counts.loc[inventory_with_counts['is_blocked'], 'blocked_event_count'], output_dir / 'blocked_events_per_crossing_histogram.png')
save_bar(reason_counts.head(10), 'Blocked event reasons', 'Reason', 'Count', output_dir / 'blocked_event_reasons.png', rotation=30)
save_bar(state_counts.head(15), 'Blocked events by state', 'State', 'Blocked event count', output_dir / 'blocked_events_by_state.png', rotation=0)
save_group_boxplot(
    to_numeric(inventory_with_counts.loc[inventory_with_counts['is_blocked'], 'Annual Average Daily Traffic Count']),
    to_numeric(inventory_with_counts.loc[~inventory_with_counts['is_blocked'], 'Annual Average Daily Traffic Count']),
    'AADT by crossing status',
    'Annual Average Daily Traffic Count',
    output_dir / 'aadt_blocked_vs_unblocked.png',
    log_scale=True,
)
save_group_boxplot(
    to_numeric(inventory_with_counts.loc[inventory_with_counts['is_blocked'], 'Total Daylight Thru Trains']),
    to_numeric(inventory_with_counts.loc[~inventory_with_counts['is_blocked'], 'Total Daylight Thru Trains']),
    'Daylight train volume by crossing status',
    'Total Daylight Thru Trains',
    output_dir / 'daylight_trains_blocked_vs_unblocked.png',
    log_scale=True,
)
save_bar(
    state_density.head(15).set_index('State Name')['blocked_crossing_share_pct'],
    'Blocked crossing share by state',
    'State',
    'Blocked crossings / total crossings (%)',
    output_dir / 'blocked_crossing_share_by_state.png',
    rotation=0,
)


The key pattern is sparse but heavy-tailed: most affected crossings have only a few blocked events, while a small number dominate the total.

In [15]:
target_crossing_id = '859522Y'
target_start_day = pd.Timestamp('2025-03-13')
target_end_day = pd.Timestamp('2025-03-15')

plot_data = blocked.loc[
    (blocked['Crossing ID'] == target_crossing_id)
    & (blocked['Date/Time'].dt.normalize() >= target_start_day)
    & (blocked['Date/Time'].dt.normalize() <= target_end_day)
].copy()

if plot_data.empty:
    print(f'No blocked-crossing rows found for {target_crossing_id} on {target_start_day.date()}')
else:
    plot_data = plot_data.groupby('Date/Time', as_index=False).size().rename(columns={'size': 'row_count'})
    fig = px.line(
        plot_data,
        x='Date/Time',
        y='row_count',
        markers=True,
        title=f'Blocked-crossing rows for {target_crossing_id} on {target_start_day.date()}',
    )
    fig.update_layout(
        xaxis_title='Date/Time',
        yaxis_title='Count of rows',
        template='plotly_white',
    )
    fig.show()


In [12]:
blocked.loc[blocked['Crossing ID'] == '859522Y']

,Crossing ID,City,State,Street,County,Railroad,Date/Time,Duration,Reason,Immediate Impacts,Additional Comments
21767,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-12-17 08:31:00,0-15 minutes,A stationary train,NaN,NaN
21768,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-12-16 14:47:00,2-6 hours,A stationary train,NaN,NaN
21769,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-12-16 14:44:00,1-2 hours,A stationary train,NaN,NaN
21770,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-12-16 11:05:00,31-60 minutes,A stationary train,NaN,NaN
21771,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-12-16 08:30:00,0-15 minutes,A stationary train,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
22699,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-01-05 04:17:00,1-2 hours,A stationary train,First responders were observed being unable to...,"Train stopped, blocking many intersections in ..."
22700,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-01-04 02:15:00,16-30 minutes,A stationary train,NaN,NaN
22701,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-01-04 01:50:00,31-60 minutes,A stationary train,First responders were observed being unable to...,"Train stopped, blocking many intersections in ..."
22702,859522Y,HOUSTON,TX,EASTWOOD STREET,HARRIS,UP,2025-01-04 01:50:00,31-60 minutes,A stationary train,First responders were observed being unable to...,NaN
